# Agentic RAG: set up, langsmith tracing

**Placement:** After the introductory FAISS RAG notebook and **before** the Tools & APIs lecture.

**Learning goals:** (1) build an agent that *decides whether to retrieve* company documents, (2) extend that agent with a simple Python tool, and (3) observe which nodes run.

We reuse the fictional **AcmeOrbit** internal policies. **FAISS is our vector store.** The second example adds a simple **calculator tool** (no REST API or server yet). Both examples use explicit **LangGraph nodes and edges**. Run cells top to bottom. Requires an OpenAI API key and API access.

## 0. Install and configure

In [2]:
#%pip install -q langgraph langchain-openai langchain-community langchain-core faiss-cpu python-dotenv
%uv add langgraph

/Users/marcelogomesmarques/Projects/LLM/master_rag/.venv/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
from dotenv import load_dotenv
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI #, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv(override=True)

model = os.getenv("OPENAI_MODEL", "qwen2.5:7b")

llm = ChatOpenAI(
            model=model,  #  "tencent/hy3:free",  "gpt-4.1-mini"
            base_url="http://localhost:11434/v1", #"https://openrouter.ai/api/v1",
            api_key="olama", # os.getenv("OPENROUTER_API_KEY"),
            temperature=0
            )

## 1. Create a tiny FAISS vector store

In [5]:
documents = [
    Document(page_content="AcmeOrbit IT policy: Employees must return a borrowed laptop within 14 calendar days after the loan ends.", metadata={"source": "IT-LAPTOP-01"}),
    Document(page_content="AcmeOrbit travel policy: Expense claims must be submitted within 30 calendar days after travel.", metadata={"source": "TRAVEL-02"}),
    Document(page_content="AcmeOrbit security policy: Employees must use multifactor authentication for company accounts.", metadata={"source": "SECURITY-03"}),
]

embeddings = OllamaEmbeddings(model="mxbai-embed-large:latest", base_url="http://localhost:11434")

vectorstore = InMemoryVectorStore.from_documents(documents, embeddings)
print(f"Vector store size: {len(vectorstore.store)} documents")

def retrieve_context(query: str, k: int = 1) -> str:
    hits = vectorstore.similarity_search(query, k=k)
    return "\n\n".join(f"Source: {d.metadata['source']}\n{d.page_content}" for d in hits)


Vector store size: 3 documents


## 2. Example A: simple **agentic RAG**

In ordinary RAG, retrieval happens **every time**. In this agentic version, the model first chooses whether retrieval is needed. The graph is:

`START → decide → (retrieve → answer | answer) → END`

The `decide` node returns a structured `needs_retrieval` boolean. For this teaching demo, the system explicitly defines which questions need the fictional company knowledge. The retrieval node performs FAISS search, and the answer node receives the retrieved context **only if retrieval occurred**.

In [13]:
from typing import TypedDict
from pydantic import BaseModel, Field

class RouteDecision(BaseModel):
    needs_retrieval: bool = Field(description="True if the question requires AcmeOrbit internal policy facts")

class RAGState(TypedDict):
    question: str
    needs_retrieval: bool
    context: str
    answer: str

router = llm.with_structured_output(RouteDecision)

def decide_node(state: RAGState):
    choice = router.invoke([
        ("system", "Set needs_retrieval=true for questions about AcmeOrbit's internal policies. Set false for general knowledge or greetings."),
        ("human", state["question"]),
    ])
    print("NODE decide → needs_retrieval:", choice.needs_retrieval)
    return {"needs_retrieval": choice.needs_retrieval}

def route_after_decide(state: RAGState):
    return "retrieve" if state["needs_retrieval"] else "answer"

def retrieve_node(state: RAGState):
    context = retrieve_context(state["question"])
    print("NODE retrieve →", context.replace("\n", " | "))
    return {"context": context}

def answer_node(state: RAGState):
    print("NODE answer")
    if state["needs_retrieval"]:
        system = ("Answer using ONLY the supplied company policy context. "
                  "If the context does not explicitly answer the question, say 'I don't know'. Cite the source ID.")
        human = f"Context:\n{state.get('context', '')}\n\nQuestion: {state['question']}"
    else:
        system = "Answer this general question concisely. You have no company policy context."
        human = state["question"]
    response = llm.invoke([("system", system), ("human", human)])
    return {"answer": response.content}

# build the agentic RAG graph
builder_a = StateGraph(RAGState)
builder_a.add_node("decide", decide_node)
builder_a.add_node("retrieve", retrieve_node)
builder_a.add_node("answer", answer_node)
builder_a.add_edge(START, "decide")
builder_a.add_conditional_edges("decide", route_after_decide, {"retrieve": "retrieve", "answer": "answer"})
builder_a.add_edge("retrieve", "answer")
builder_a.add_edge("answer", END)
agentic_rag = builder_a.compile()

print(agentic_rag.get_graph().draw_mermaid())

png_data = agentic_rag.get_graph().draw_mermaid_png()
with open("agentic_rag_graph.png", "wb") as f:
    f.write(png_data)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	decide(decide)
	retrieve(retrieve)
	answer(answer)
	__end__([<p>__end__</p>]):::last
	__start__ --> decide;
	decide -.-> answer;
	decide -.-> retrieve;
	retrieve --> answer;
	answer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



 Run the agentic flow

In [12]:
questions = [
    "How many days do employees have to return a borrowed laptop?",
    "What is 2 + 2?",
]

for question in questions:
    result = agentic_rag.invoke(
        {
            "question": question,
            "needs_retrieval": False,
            "context": "",
            "answer": "",
        },
        config={"recursion_limit": 8},
    )

    print("\nQUESTION:", question)
    print("ANSWER:", result["answer"])

NODE decide → needs_retrieval: True
NODE retrieve → Source: IT-LAPTOP-01 | AcmeOrbit IT policy: Employees must return a borrowed laptop within 14 calendar days after the loan ends.
NODE answer

QUESTION: How many days do employees have to return a borrowed laptop?
ANSWER: Employees must return a borrowed laptop within 14 calendar days after the loan ends. Source: IT-LAPTOP-01
NODE decide → needs_retrieval: False
NODE answer

QUESTION: What is 2 + 2?
ANSWER: 2 + 2 equals 4.


**Observe:** The first question should run `decide → retrieve → answer`; the general question should skip retrieval. Routing is model-driven, so inspect the printed node trace rather than assuming every run is identical.

**Limitation:** FAISS returns nearest matches even when no document answers a question. The answer prompt instructs the model to abstain, but production systems also add relevance checks.

## 3. Example B: **agentic RAG + a tool**

Now the agent can choose between **two tools**:

- `search_company_policy`: looks up internal company policy using FAISS.

The LLM decides which tool(s) to call. LangGraph's `ToolNode` executes them. The graph is:

`START → agent → (tools → agent | END)`

This is a bridge to the next lecture, where tools will call REST APIs.

In [16]:
from datetime import date, timedelta

@tool
def search_company_policy(question: str) -> str:
    """Search AcmeOrbit internal policies. Use for any question about company rules or deadlines."""
    print("TOOL search_company_policy called with:", question)
    return retrieve_context(question)

tools = [search_company_policy]
llm_with_tools = llm.bind_tools(tools)

SYSTEM_B = ("You are an AcmeOrbit helpdesk assistant. "
            "For company policy questions, you MUST call search_company_policy before answering. "
            "Never invent company policies. Only state company-policy facts present in tool results; "
            "if absent, say 'I don't know'. Cite policy source IDs. "
            "Answer clearly and concisely.")

def agent_node(state: MessagesState):
    print("NODE agent")
    response = llm_with_tools.invoke([SystemMessage(content=SYSTEM_B)] + state["messages"])
    return {"messages": [response]}

# build the agentic RAG graph with tool usage
builder_b = StateGraph(MessagesState)
builder_b.add_node("agent", agent_node)
builder_b.add_node("tools", ToolNode(tools))
builder_b.add_edge(START, "agent")
builder_b.add_conditional_edges("agent", tools_condition)
builder_b.add_edge("tools", "agent")
agentic_rag_with_tool = builder_b.compile()
print(agentic_rag_with_tool.get_graph().draw_mermaid())

png_data = agentic_rag_with_tool.get_graph().draw_mermaid_png()
with open("agentic_rag_pluss_tools_graph.png", "wb") as f:
    f.write(png_data)


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools<hr/><small><em>__tools = search_company_policy</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Utils

In [10]:
def run_agentic_rag(question: str):
    result = agentic_rag_with_tool.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"recursion_limit": 12},
    )
    return result


def print_agentic_rag_results(question, result):
    print("-"*60)
    print("Agentic RAG Results:")
    print("\nQUESTION:", question)
    print("\nFINAL ANSWER:", result["messages"][-1].content)

    print("\nTOOL CALLS:")
    tool_called = False

    for message in result["messages"]:
        if getattr(message, "tool_calls", None):
            for call in message.tool_calls:
                print("-", call["name"], call["args"])
                tool_called = True

    if not tool_called:
        print("None")


In [11]:
print("-"*60)
q = ("An AcmeOrbit employee's borrowed laptop loan ended on 2026-10-01. "
     "According to company policy, what is the return deadline date?")
print("QUESTION:", q)

print("-"*60)
print("Invoke Agentic RAG with tools:")
result = agentic_rag_with_tool.invoke(
    {"messages": [HumanMessage(content=q)]}, config={"recursion_limit": 12})

print("-"*60)
print("Run Agentic RAG:")
result = run_agentic_rag(q)

print_agentic_rag_results(q, result)

------------------------------------------------------------
QUESTION: An AcmeOrbit employee's borrowed laptop loan ended on 2026-10-01. According to company policy, what is the return deadline date?
------------------------------------------------------------
Invoke Agentic RAG with tools:
NODE agent
TOOL search_company_policy called with: What is the laptop return deadline according to AcmeOrbit policy?
NODE agent
------------------------------------------------------------
Run Agentic RAG:
NODE agent
TOOL search_company_policy called with: What is the laptop return deadline according to AcmeOrbit policy?
NODE agent
------------------------------------------------------------
Agentic RAG Results:

QUESTION: An AcmeOrbit employee's borrowed laptop loan ended on 2026-10-01. According to company policy, what is the return deadline date?

FINAL ANSWER: According to AcmeOrbit IT policy (Source: IT-LAPTOP-01), employees must return a borrowed laptop within 14 calendar days after the loan e

In [13]:
while True:
    user_input = input("Vous: ").strip()

    # Stop the loop
    if user_input.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break

    # Ignore empty input
    if not user_input:
        continue

    # Process one question at a time
    result = run_agentic_rag(user_input)
    print_agentic_rag_results(user_input, result)

NODE agent
------------------------------------------------------------
Agentic RAG Results:

QUESTION: What is the capital of France?

FINAL ANSWER: I don't know. This question is not related to AcmeOrbit's company policies.

TOOL CALLS:
None
NODE agent
TOOL search_company_policy called with: When are travel expenses due if travel added on 2026-10-05?
NODE agent
------------------------------------------------------------
Agentic RAG Results:

QUESTION: When are travel expenses due if travel added on 2026-10-05?

FINAL ANSWER: According to AcmeOrbit's travel policy (TRAVEL-02), expense claims must be submitted within 30 calendar days after travel. Given that the travel was added on 2026-10-05, the expenses should be due by 2026-11-04.

TOOL CALLS:
- search_company_policy {'question': 'When are travel expenses due if travel added on 2026-10-05?'}
NODE agent
TOOL search_company_policy called with: Is there any requirement if vacation allowance is not present in the documents?
NODE agent

**Expected reasoning:** Retrieve `IT-LAPTOP-01` → learn the 14-calendar-day rule → call `add_days("2026-10-01", 14)` → answer **2026-10-15** with the policy source. Model-driven tool use can vary; the printed tool calls show what actually happened.

**Important:** The `add_days` function is a **local Python tool**, not a REST API. We introduce APIs in the next lecture.


**Exercises:**
1. In Example A, ask “What is the capital of France?” Verify that retrieval is skipped.
2. In Example B, ask “When are travel expenses due if travel ended on 2026-10-05?” Inspect both tool calls.
3. Ask about an AcmeOrbit vacation allowance not present in the documents. Does the agent correctly abstain? Why might a similarity threshold help?

**Next:** Replace the local calculator tool with a business REST endpoint and learn how LangGraph agents call external APIs.